# Project 1 — Issue Report Classification
## Notebook 03: Classical machine learning

**Track B deliverable.**

Four classifier families over a shared TF-IDF representation, plus the two
experiments that decide how the features are built:

1. **Preprocessing ablation** — which cleaning level actually helps.
2. **Hyperparameter search** — cross-validated, on the training split only.
3. **Final results** — the competition protocol, against the SetFit baseline.

The headline finding is in section 1, and it is not the one Track A expected.

In [ ]:
import json

import pandas as pd

from ai4se.classical import CLASSICAL_MODELS, TUNED_MODELS, TUNED_PREPROCESSING
from ai4se.evaluation import (
    SETFIT_OVERALL,
    cross_validate,
    leaderboard_from_disk,
    load_result,
)
from ai4se.loader import PROJECT_ROOT, load_split
from ai4se.preprocessing import make_cleaner

pd.set_option("display.width", 140)
TABLES = PROJECT_ROOT / "results" / "tables"

train = load_split("train", kind="memory")
test = load_split("test", kind="memory")
train.apply(make_cleaner(**TUNED_PREPROCESSING))
test.apply(make_cleaner(**TUNED_PREPROCESSING))
print(train, "|", test)
print("preprocessing:", TUNED_PREPROCESSING)

---
## 1. Does cleaning help? (Preprocessing ablation)

Track A built a three-level cleaning pipeline on the reasonable assumption that
stripping noise from issue bodies would help a bag-of-words model. That
assumption needed testing rather than believing.

Each configuration below was cross-validated with the same logistic-regression
pipeline; only the preprocessing changed.

In [ ]:
ablation = pd.DataFrame(json.loads((TABLES / "ablation_preprocessing.json").read_text()))
pivot = ablation.pivot_table(
    index=["level"], columns=["title_weight"], values="macro_f1", aggfunc="max"
)
display(ablation.head(8).reset_index(drop=True))
print("\nbest macro F1 by cleaning level and title weight:")
display(pivot.round(4))

**`full` cleaning is the worst of the three levels**, by roughly 0.015 macro F1
— about half a standard deviation of the fold-to-fold spread, and consistent
across every title weight and truncation setting.

The explanation is in what `full` removes. Stop-word removal deletes *would*,
*could*, *should* and *please*; lemmatisation collapses tense. Those are
precisely the modal and evaluative words that Track A's own term analysis
identified as the signature of a feature request. The cleaning was removing the
signal along with the noise.

`light` cleaning — which strips code blocks, stack traces, markup and URLs but
leaves natural language intact — is what the rest of Track B uses.

**Two lessons worth stating in the report.** First, a preprocessing step that
is obviously sensible can still be harmful, and only an ablation shows it.
Second, Track A's own analysis contained the warning: code blocks were far more
common in bug reports than in feature requests, which meant stripping them was
never going to be free.

### Title weighting and truncation

In [ ]:
display(
    ablation.pivot_table(index="level", columns="max_words", values="macro_f1", aggfunc="max")
    .round(4)
)

Repeating the title three times gives a small, consistent gain: titles are
short and dense with exactly the vocabulary that distinguishes the classes.
Truncation past 200 words neither helps nor hurts much, which matches the
length distribution from Track A — the median issue is far shorter than any of
these thresholds, so the setting only affects the tail.

---
## 2. Hyperparameter search

Cross-validated on the **training split only**. The test split is not consulted
until section 3.

In [ ]:
grid = json.loads((TABLES / "grid_search.json").read_text())
for name, rows in grid.items():
    print(f"=== {name} ===")
    display(pd.DataFrame(rows).head(5).reset_index(drop=True))

Bigrams win consistently, and that is the only setting whose advantage clearly
exceeds the fold-to-fold noise. Everything else is close: the best and fifth-best
configurations differ by about 0.002 macro F1 against a standard deviation of
roughly 0.029.

**An honest note for the report.** The grid was first run under `full` cleaning;
when the ablation moved the pipeline to `light`, the winning `min_df` changed.
Hyperparameters and preprocessing are not independent, and re-tuning after
altering the pipeline is not optional. The two tuned models also disagree —
logistic regression prefers `min_df=2`, the SVM `min_df=1` — which is a
coin-flip within the noise rather than a finding.

---
## 3. Cross-validation on the training split

In [ ]:
rows = []
for name, factory in {**CLASSICAL_MODELS, **TUNED_MODELS}.items():
    result = cross_validate(factory, train, k=10, model_name=name)
    rows.append({
        "model": name,
        "macro F1": result.mean_macro_f1,
        "std": result.std_macro_f1,
        "macro AUC": result.mean_macro_auc,
    })
pd.DataFrame(rows).set_index("model").sort_values("macro F1", ascending=False).round(4)

`LinearSVC` has no probability head, so its AUC is `NaN`. That is deliberate:
wrapping it in probability calibration would change the model being measured.
The harness reports `None` rather than approximating an AUC from hard labels.

Random forest trails the linear models, which is expected — trees cope poorly
with the very high-dimensional sparse features TF-IDF produces. It is a finding,
not a bug.

---
## 4. Final results on the competition protocol

In [ ]:
board = leaderboard_from_disk(TABLES)
classical = board[board.index.str.contains("TF-IDF|SetFit")]
display(classical)
print(f"best classical: {classical['overall'].iloc[1]:.4f}  "
      f"(baseline {SETFIT_OVERALL})")

The best classical configuration reaches **0.7603**, leaving a gap of 0.0667 to
the SetFit baseline.

Note that `bitcoin/bitcoin` and `microsoft/vscode` are hard for every model,
which mirrors the baseline's own per-repository spread — SetFit also scores
lowest on `bitcoin/bitcoin` (0.7555). Project difficulty is a property of the
data, not of any one approach.